# `hoi` — Higher-Order Information

> **Docs**: https://brainets.github.io/hoi/

**`hoi`** estimates higher-order information measures from **continuous** data using the KSG (Kozachenko–Leonenko) k-nearest-neighbour estimator.
No binning is required: the library works directly on raw floating-point samples.

### What we compute
| Measure | Symbol | Interpretation |
|---------|--------|---------------|
| O-information | $\Omega = TC - DTC$ | $>0$ redundancy, $<0$ synergy |
| Total Correlation | $TC$ | Total shared information |
| Dual Total Correlation | $DTC$ | Higher-order unique information |

### Datasets
* **Artificial** — controlled redundant and synergistic triplets.
* **Real** — UCI Wine dataset (top-5 features, all multiplets of size 3–5).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from numpy import random
from itertools import combinations
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.feature_selection import mutual_info_classif

from hoi.metrics import Oinfo, TC, DTC

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
print("Imports OK.")


## 1. Artificial Data

In [ ]:
n_features, n_samples = 3, 500
eta = 0.1

source = random.normal(0, 1, n_samples)
data_red = np.array([
    source,
    source + random.normal(0, 1, n_samples) * eta,
    source + random.normal(0, 1, n_samples) * eta,
])

s0 = random.normal(0, 1, n_samples)
s1 = random.normal(0, 1, n_samples)
data_syn = np.array([
    s0 + random.normal(0, 1, n_samples) * eta,
    s1 + random.normal(0, 1, n_samples) * eta,
    s0 + s1 + random.normal(0, 1, n_samples) * eta,
])

# hoi expects (n_samples, n_features)
X_red = data_red.T   # (500, 3)
X_syn = data_syn.T   # (500, 3)
print("Input shapes:", X_red.shape, X_syn.shape)


## 2. O-information

Positive $\Omega$ → redundancy-dominated; negative $\Omega$ → synergy-dominated.


In [ ]:
model_oinfo_red = Oinfo(X_red)
model_oinfo_syn = Oinfo(X_syn)

oinfo_red = model_oinfo_red.fit(minsize=3, maxsize=3)
oinfo_syn = model_oinfo_syn.fit(minsize=3, maxsize=3)

omega_red = float(np.asarray(oinfo_red).ravel()[0])
omega_syn = float(np.asarray(oinfo_syn).ravel()[0])

print(f"O-information  Ω(X0,X1,X2)")
print(f"  Redundant:    {omega_red:+.4f}   ← POSITIVE ✓")
print(f"  Synergistic:  {omega_syn:+.4f}   ← NEGATIVE ✓")


## 3. Total Correlation and Dual Total Correlation

In [ ]:
# Total Correlation
tc_red = float(np.asarray(TC(X_red).fit(minsize=3, maxsize=3)).ravel()[0])
tc_syn = float(np.asarray(TC(X_syn).fit(minsize=3, maxsize=3)).ravel()[0])

# Dual Total Correlation
dtc_red = float(np.asarray(DTC(X_red).fit(minsize=3, maxsize=3)).ravel()[0])
dtc_syn = float(np.asarray(DTC(X_syn).fit(minsize=3, maxsize=3)).ravel()[0])

print(f"{'Measure':<8} {'Redundant':>12} {'Synergistic':>14}")
print("-" * 36)
print(f"{'TC':<8} {tc_red:>12.4f} {tc_syn:>14.4f}")
print(f"{'DTC':<8} {dtc_red:>12.4f} {dtc_syn:>14.4f}")
print(f"{'Ω=TC-DTC':<8} {tc_red-dtc_red:>+12.4f} {tc_syn-dtc_syn:>+14.4f}")
print()
print("Verification: Ω should equal TC - DTC")
print(f"  Redundant:    direct={omega_red:+.4f}  TC-DTC={tc_red-dtc_red:+.4f}")
print(f"  Synergistic:  direct={omega_syn:+.4f}  TC-DTC={tc_syn-dtc_syn:+.4f}")


## 4. Visualisation — Artificial Data

In [ ]:
measures = ['TC', 'DTC', '$\\Omega$ = TC$-$DTC']
r_vals = [tc_red, dtc_red, omega_red]
s_vals = [tc_syn, dtc_syn, omega_syn]

x = np.arange(len(measures))
w = 0.35
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - w/2, r_vals, w, label='Redundant',   color=sns.color_palette()[0])
ax.bar(x + w/2, s_vals, w, label='Synergistic', color=sns.color_palette()[2])
ax.axhline(0, color='black', lw=0.8, ls='--')
ax.set_xticks(x); ax.set_xticklabels(measures, fontsize=12)
ax.set_ylabel('Value (bits)')
ax.set_title('hoi: TC, DTC, and O-information\nfor redundant vs synergistic triplets',
             fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()


## 5. Real Data — UCI Wine Dataset

We apply `hoi` to all multiplets of size 3–5 from the **top-5 features** (ranked by MI with class label).


In [ ]:
wine = load_wine()
X_wine, y_wine = wine.data, wine.target
feature_names = list(wine.feature_names)

mi_scores = mutual_info_classif(X_wine, y_wine, random_state=SEED)
order = np.argsort(mi_scores)[::-1]
top5 = [feature_names[i] for i in order[:5]]
X_top5 = X_wine[:, order[:5]]   # (178, 5)
print("Top-5 features:", top5)

model_wine = Oinfo(X_top5)
oinfo_wine = model_wine.fit(minsize=3, maxsize=5)
oinfo_arr = np.asarray(oinfo_wine).ravel()

print(f"\nMultiplets evaluated: {len(oinfo_arr)}")
print(f"Ω range: [{oinfo_arr.min():.4f}, {oinfo_arr.max():.4f}]")
print(f"Redundant (Ω>0): {np.sum(oinfo_arr > 0)} / {len(oinfo_arr)}")
print(f"Synergistic (Ω<0): {np.sum(oinfo_arr < 0)} / {len(oinfo_arr)}")


In [ ]:
# Identify most redundant and most synergistic multiplets
all_combos = []
for k in range(3, 6):
    all_combos += list(combinations(range(5), k))

df = pd.DataFrame({"features_idx": all_combos, "o": oinfo_arr})
df["features"] = df["features_idx"].apply(lambda idxs: [top5[i] for i in idxs])

print("Most REDUNDANT multiplet:")
row = df.loc[df["o"].idxmax()]
print(" ", row["features"], f"  Ω = {row['o']:+.4f}")

print("\nMost SYNERGISTIC multiplet:")
row = df.loc[df["o"].idxmin()]
print(" ", row["features"], f"  Ω = {row['o']:+.4f}")


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(oinfo_arr, bins=20, color=sns.color_palette()[0],
        edgecolor='white', linewidth=0.5)
ax.axvline(0, color='black', lw=1.5, ls='--', label=r'$\Omega=0$')
ax.axvline(oinfo_arr.mean(), color='tomato', lw=1.5, ls='-',
           label=f'Mean = {oinfo_arr.mean():.3f}')
ax.set_xlabel(r'O-information $\Omega$ (bits)')
ax.set_ylabel('Count')
ax.set_title('hoi: O-information distribution over all multiplets\n(Wine top-5 features, orders 3–5)',
             fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()
